# Week 43-44: Paired Challenge Set

In [1]:
import json
from pathlib import Path

import pandas as pd

from ku_nlp_group_25_project.baselines.lexical_overlap_classifier import LexicalOverlapClassifier
from ku_nlp_group_25_project.constants import LANGUAGES

outputs_dir = Path.cwd().parent / "outputs" / "week_43_44"
outputs_dir.mkdir(exist_ok=True, parents=True)

challenge = pd.DataFrame(json.loads((Path.cwd().parent / "data" / "challenge.json").read_text()))

languages = LANGUAGES.copy()
languages.append("zh") # include chinese questions

/home/kieran/code/github.com/kieranroneill/ku-nlp-group-25-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Baselines

In [2]:
baselines_outputs_dir = outputs_dir / "baselines"
baselines_outputs_dir.mkdir(exist_ok=True, parents=True)

### 1.2. Baseline 2: Deterministic rule-based classification

In [3]:
classifier = LexicalOverlapClassifier(
    casefolding=True,
    languages=languages,
    min_shared_tokens=1,
    normalisation_form="NFC",
)
predictions = classifier.predictions(
    dataframe=challenge,
)
metrics = classifier.evaluate(
    predictions_dataframe=predictions,
)

predictions["correct"] = predictions["answerable"] == predictions["predicted_answerable"]
paired = (
    predictions.groupby("pair_id", as_index=False)
    .agg(lang=("lang", "first"), n_answered=("predicted_answerable", "sum"),
         pair_correct=("correct", "all"))
)
paired["n_records"] = predictions.groupby("pair_id")["id"].count().values

predictions.to_csv(baselines_outputs_dir / "rule_based_classifier_predictions.csv", index=False)
paired.to_csv(baselines_outputs_dir / "rule_based_classifier_paired.csv", index=False)
metrics.to_csv(baselines_outputs_dir / "rule_based_classifier_metrics.csv", index=False)
